# 관통 프로젝트 3단계 (풀이)

검증용 풀이 노트북입니다. 공유하지 않습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(1장의 공개 LLM, 임베딩 모델)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [1]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

사용하는 API: anthropic


In [2]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install sentence-transformers rank-bm25 pypdf
# 8장과 관통 프로젝트 3단계에서 씁니다. 문서를 읽고, 청크로 나누고, 검색하는 함수는 같은 폴더의 corpus.py에 있습니다
import csv
import random
import re

import numpy as np

import corpus
import embed

with open(f'{DATA}/문의.csv', encoding='utf-8') as file:
    inquiries = list(csv.DictReader(file))        # 문의 60건. 관통 프로젝트 1, 2단계의 코드가 씁니다
LABELS = ['설비 이상', '품질', '안전', '문서와 데이터 요청']

# 100만 토큰당 단가(달러). 5장과 같습니다(llm.py가 쓰는 Anthropic 모델, 2026-11 확인)
PRICES = {'소형': {'input': 0.10, 'output': 0.50}, '기본': {'input': 2.00, 'output': 10.00},
          '대형': {'input': 4.00, 'output': 20.00}}


def cost(reply, tier):
    """응답 하나의 요금(달러). 입력 토큰과 출력 토큰(thinking 포함)에 각각의 단가를 곱해 더합니다"""
    price = PRICES[tier]
    return (reply.input_tokens * price['input'] + reply.output_tokens * price['output']) / 1_000_000


catalog = corpus.catalog()                               # {문서 id: 메타데이터}. 문서 120건
questions = {item['id']: item for item in corpus.evalset()}   # 평가 질문 62개
chunks, failed = corpus.load_chunks()                    # 문서마다 파싱하고 절 단위 청크로 나눕니다(corpus.py)
searcher = corpus.Search(chunks)                         # 청크마다 임베딩과 BM25 인덱스를 만듭니다. 1~2분 걸립니다


def is_valid(chunk):
    """메타데이터 필터. 폐기된 구판의 청크를 검색 대상에서 뺍니다"""
    return chunk['status'] == '유효'


RAG_PROMPT = """아래 [검색 결과]만 근거로 질문에 답하세요.
- 문장마다 근거가 된 청크의 번호를 [SOP-WB-001#3]처럼 적습니다.
- 검색 결과에 답이 없으면 "검색한 문서에서 찾지 못했습니다"라고만 답합니다.
- 검색 결과끼리 내용이 다르면 날짜가 늦은 문서를 따르고, 내용이 다르다는 것을 함께 적습니다.

[검색 결과]
{context}

[질문]
{question}"""


def rag_prompt(question, hits, with_meta=True):
    """검색한 청크를 번호와 함께 RAG_PROMPT에 넣습니다. with_meta가 참이면 청크마다 상태, 개정, 날짜를 붙입니다"""
    blocks = []
    for hit in hits:
        meta = f" (상태 {hit['status']}, 개정 {hit['revision']}, {hit['date']})" if with_meta else ''
        blocks.append(f"[{hit['chunk_id']}]{meta}\n{hit['text']}")
    return RAG_PROMPT.format(context='\n\n'.join(blocks), question=question)


print(f'문서 {len(catalog)}건, 청크 {len(chunks)}개, 글을 뽑지 못한 문서 {failed}')


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


임베딩 모델: intfloat/multilingual-e5-small (PC에 저장된 파일을 씁니다)


문서 120건, 청크 487개, 글을 뽑지 못한 문서 ['EHS-010']
소형 모델: claude-haiku-5-5
기본 모델: claude-sonnet-5-5
대형 모델: claude-opus-5-5


준비 완료


오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 앞 단계의 코드: 1단계 풀이

In [3]:
SYSTEM = '너는 반도체 후공정 라인의 문의에 답하는 도우미다. 사내 정보를 모르면 지어내지 말고 모른다고 답한다.'


def answer(question, tier='소형', tries=3, send=llm.send, sleep=time.sleep):
    """질문 하나에 답합니다. 돌려주는 dict의 키: question, status('완료', '확인 필요', '실패'),
    answer, input_tokens, output_tokens, cost, reason"""
    result = {'question': question, 'status': '실패', 'answer': '', 'input_tokens': 0,
              'output_tokens': 0, 'cost': 0.0, 'reason': ''}
    for attempt in range(tries):
        try:
            reply = send([{'role': 'user', 'content': question}], system=SYSTEM, model=tier, max_tokens=2000)
        except llm.CallError as error:
            result['reason'] = str(error)
            if not error.retryable:
                return result                         # 요청을 고쳐야 풀리는 오류는 다시 보내지 않습니다
            if attempt < tries - 1:
                sleep(2 ** attempt + random.random())  # 1초, 2초 … 백오프에 지터를 더합니다
            continue
        result.update(answer=reply.text, input_tokens=reply.input_tokens, output_tokens=reply.output_tokens,
                      cost=cost(reply, tier), reason=reply.stop_reason,
                      status='완료' if reply.stop_reason == 'end' else '확인 필요')
        return result
    return result


def mock_send(results):
    """미리 정해 둔 결과를 차례로 돌려주는 mock. 결과가 예외이면 발생시킵니다"""
    queue = list(results)

    def send(messages, **options):
        result = queue.pop(0)
        if isinstance(result, Exception):
            raise result
        return result
    return send


done = llm.Reply(text='완료된 답', input_tokens=50, output_tokens=20, stop_reason='end', model='mock', seconds=0.1)
cut = llm.Reply(text='잘린 답', input_tokens=50, output_tokens=2000, stop_reason='max_tokens', model='mock', seconds=0.1)
busy, missing = llm.CallError('rate_limit', 429, '요청이 많음'), llm.CallError('not_found', 404, '없는 모델')
no_wait = lambda seconds: None                       # 테스트에서는 기다리지 않습니다
print('429 두 번 뒤 성공:', answer('질문', send=mock_send([busy, busy, done]), sleep=no_wait)['status'])
print('429 세 번:', answer('질문', send=mock_send([busy, busy, busy]), sleep=no_wait)['status'])
print('404 한 번(다음 응답은 성공):', answer('질문', send=mock_send([missing, done]), sleep=no_wait)['status'])
cut_result = answer('질문', send=mock_send([cut]), sleep=no_wait)
print('잘린 응답:', cut_result['status'], cut_result)

results = [answer(item['문의']) for item in inquiries if item['번호'] in ('Q31', 'Q33', 'Q35', 'Q36', 'Q38')]
for result in results:
    print(f"\n[{result['status']}] {result['question']}\n  → {result['answer'].strip()[:100]}")
print(f"\n요금 합계 {sum(result['cost'] for result in results):.5f}달러, "
      f"실패와 확인 필요 목록 {[(result['question'], result['reason']) for result in results if result['status'] != '완료']}")

429 두 번 뒤 성공: 완료
429 세 번: 실패
404 한 번(다음 응답은 성공): 실패
잘린 응답: 확인 필요 {'question': '질문', 'status': '확인 필요', 'answer': '잘린 답', 'input_tokens': 50, 'output_tokens': 2000, 'cost': 0.001005, 'reason': 'max_tokens'}



[완료] 와이어 본딩 작업 표준 최신본은 어디에 있나요?
  → 와이어 본딩 작업 표준의 최신본 위치는 제가 확인할 수 없습니다. 사내 문서 시스템에 접근할 수 없고, 문서 위치나 버전을 추측해서 말씀드리면 잘못된 문서를 참고하실 수 있어 위험

[완료] 몰딩 컴파운드 보관 조건이 적힌 문서를 찾고 있습니다.
  → 사내 문서 저장소나 검색 시스템에는 접근할 수 없어서, 해당 문서를 직접 찾아드릴 수는 없습니다. 사내 문서 관리 체계에 있는 실제 보관 조건을 확인해 주셔야 합니다.

**문서를

[완료] 신규 제품 QFN-48의 검사 기준서를 공유해 주세요.
  → 죄송하지만 저는 사내 문서 저장소나 문서 관리 시스템에 접근할 수 없어서, QFN-48 검사 기준서를 직접 공유해 드릴 수가 없습니다. 기준서 내용을 임의로 만들어 드리면 실제 라

[완료] 3월 품질 회의 회의록을 받을 수 있을까요?
  → 죄송하지만 저는 사내 문서 시스템에 접근할 수 없어서 3월 품질 회의록을 찾아 드릴 수 없습니다. 회의록 내용을 임의로 추정해서 말씀드리는 것도 정확하지 않을 수 있어 드리지 않겠

[완료] 교대 인수인계 양식 파일이 어디 있는지 알려 주세요.
  → 죄송하지만 사내 파일 위치나 공유 드라이브 경로는 제가 알 수 없습니다. 제가 가진 정보에 없는 내용이라 추측으로 경로를 말씀드리면 잘못된 파일을 찾으실 수 있어서요.

아래 순서

요금 합계 0.00152달러, 실패와 확인 필요 목록 []


## 앞 단계의 코드: 2단계 풀이

In [4]:
import functools

ANSWER_SCHEMA = {'type': 'object', 'additionalProperties': False,
                 'required': ['category', 'urgent', 'needs_document', 'answer'],
                 'properties': {'category': {'type': 'string', 'enum': LABELS}, 'urgent': {'type': 'boolean'},
                                'needs_document': {'type': 'boolean'}, 'answer': {'type': 'string'}}}


def check(data):
    """업무 규칙 검사. 문제가 없으면 None, 있으면 사유 문자열을 돌려줍니다"""
    if data['category'] == '문서와 데이터 요청' and not data['needs_document']:
        return '문서 요청인데 사내 문서가 필요하지 않다고 답함'
    if data['category'] == '안전' and not data['urgent']:
        return '안전 문의인데 긴급이 아니라고 답함'
    return None


def answer_structured(question):
    """1단계의 answer()에 구조화 출력을 더합니다. 결과 dict에 data(파싱한 dict)와 status, reason을 넣습니다"""
    result = answer(question, send=functools.partial(llm.send, schema=ANSWER_SCHEMA))
    result['data'] = None
    if result['status'] != '완료':                   # 잘림과 거절은 스키마에 맞지 않을 수 있으므로 파싱하지 않습니다
        result['status'] = '실패'
        return result
    result['data'] = json.loads(result['answer'])
    problem = check(result['data'])
    result['status'], result['reason'] = ('사람 확인', problem) if problem else ('자동 처리', '')
    return result


for item in [item for item in inquiries if item['번호'] in ('Q21', 'Q24', 'Q31', 'Q43', 'Q49', 'Q56')]:
    result = answer_structured(item['문의'])
    print(f"[{result['status']}] {item['번호']} {item['문의']}\n  → {result['data']}, 사유: {result['reason']}, 요금 {result['cost']:.5f}달러")

[자동 처리] Q21 클린룸 B구역 바닥에 약품이 흘러 있는데 무슨 약품인지 모르겠습니다.
  → {'category': '안전', 'urgent': True, 'needs_document': False, 'answer': '즉시 해당 구역 접근을 막고 주변 작업자를 대피시키세요. 흘러나온 약품의 종류를 모르는 상태에서는 맨손이나 일반 걸레로 닦지 말고, 보호장비(보호장갑, 보안경, 필요 시 호흡보호구)를 갖춘 담당자만 처리하도록 하세요. 가능하면 용기나 라벨, 주변 MSDS(물질안전보건자료)를 확인해 성분을 파악하고, 확인이 어려우면 즉시 안전관리 담당자 또는 환경안전 부서에 연락하세요. 사내 세부 절차와 연락처는 제가 알지 못하니 현장 안전 매뉴얼을 따라 주세요.'}, 사유: , 요금 0.00020달러


[자동 처리] Q24 비상구 앞에 자재 카트가 쌓여 통로가 막혔습니다.
  → {'category': '안전', 'urgent': True, 'needs_document': False, 'answer': '비상구 앞 통로가 막힌 것은 화재나 비상 시 대피를 방해할 수 있는 위험 상황이므로 즉시 조치가 필요합니다. 해당 구역에서 자재 카트를 안전한 곳으로 치우고 비상구 앞 통로를 확보해 주시고, 주변 작업자에게 통로 정리 사실을 알려 주세요. 반복적으로 쌓이는 원인이 있는지 확인하고, 담당 안전 관리자 또는 현장 관리자에게 바로 보고하시기 바랍니다. 사내 구체적인 보고 절차와 연락처는 제가 알지 못하므로 현장 안전 담당 부서에 확인해 주세요.'}, 사유: , 요금 0.00019달러


[자동 처리] Q31 와이어 본딩 작업 표준 최신본은 어디에 있나요?
  → {'category': '문서와 데이터 요청', 'urgent': False, 'needs_document': True, 'answer': "와이어 본딩 작업 표준의 최신본 위치는 제가 확인할 수 없습니다. 사내 문서 관리 시스템(품질 또는 공정 문서 관리 부서)에서 '와이어 본딩 작업 표준'으로 검색하거나, 담당 엔지니어 또는 품질 부서에 최신 승인본을 요청해 주세요."}, 사유: , 요금 0.00013달러


[자동 처리] Q43 다이싱 소 D-01 근처에서 냉각수가 바닥으로 새고 있습니다.
  → {'category': '안전', 'urgent': True, 'needs_document': False, 'answer': '냉각수 누수는 감전 및 미끄럼 사고 위험이 있으니 즉시 조치가 필요합니다. 다이싱 소 D-01의 전원을 차단하고 (가능하면 비상정지 버튼 사용), 누수 구역 주변에 접근을 통제하며 미끄럼 주의 표지를 설치하십시오. 누수가 전기 설비나 배선에 닿지 않도록 주의하고, 흡수 패드나 걸레 등으로 바닥 물기를 우선 제거하십시오. 이후 현장 담당자 또는 설비 보전팀에 즉시 보고해 누수 원인(배관, 실링, 노즐 연결부 등)을 점검받으시기 바랍니다. 사내 세부 절차(보고 체계, 연락처, 점검 매뉴얼)는 제가 알지 못하니 해당 부서의 안내를 따라 주십시오.'}, 사유: , 요금 0.00022달러


[자동 처리] Q49 마킹 불량 제품이 출하 박스에 섞였는지 확인할 출하 기록이 필요합니다.
  → {'category': '문서와 데이터 요청', 'urgent': True, 'needs_document': True, 'answer': '마킹 불량 제품이 출하 박스에 섞였는지 확인하려면 해당 출하 건의 출하 기록(박스 번호, 제품 시리얼/로트 번호, 마킹 검사 결과, 출하 일시 등)이 필요합니다. 제가 사내 시스템에 직접 접근할 수 없어 기록 내용은 확인해 드릴 수 없으니, MES나 출하 관리 시스템에서 해당 로트의 출하 이력과 마킹 검사 기록을 조회해 주시거나 담당 부서에 요청해 주시기 바랍니다. 불량 의심 제품이 이미 출하되었을 가능성이 있으므로 품질 담당자에게 즉시 공유하는 것을 권장합니다.'}, 사유: , 요금 0.00020달러


[자동 처리] Q56 본더 한 대가 또 멈췄습니다.
  → {'category': '설비 이상', 'urgent': True, 'needs_document': False, 'answer': '본더 정지 문의로 판단됩니다. 현재 사내 설비 로그나 알람 정보는 확인할 수 없어 원인을 특정할 수 없습니다. 우선 정지 시점에 표시된 알람 코드와 메시지, 직전 작업 공정과 웨이퍼/리드프레임 상태를 확인해 주세요. 반복 정지라면 생산 영향을 줄이기 위해 해당 설비 담당 엔지니어나 설비 보전팀에 즉시 연락해 점검을 요청하고, 정확한 조치 기준은 사내 설비 매뉴얼이나 담당자 확인을 따르시기 바랍니다.'}, 사유: , 요금 0.00018달러


## 실습 P3. 문서를 검색해 근거를 붙여 답하기

교재의 절: "관통 프로젝트 3단계"

`answer_with_documents(question, k)`를 작성합니다. 유효한 문서에서 청크 k개를 검색하고(`searcher.search`, `is_valid`), `rag_prompt()`로 만든 프롬프트를 1단계의 `answer()`로 보냅니다. 결과 dict에 검색한 청크 번호(`hits`)와 답에 적힌 출처 번호(`sources`)를 더합니다. 상태가 "완료"인 답 가운데, 찾지 못했다는 답이 아닌데 출처가 하나도 없거나, 검색하지 않은 청크 번호를 출처로 적었으면 "확인 필요"로 바꾸고 사유를 적습니다. 평가 질문 5개의 상태, 답, 출처가 출력되면 끝납니다. 이 셀은 API를 5번 호출합니다.

완료 기준

- 질문 5개의 상태, 답 앞부분, 검색한 청크, 출처가 출력됩니다.
- 답이 없는 질문(E55)은 찾지 못했다는 답이 나오고 "확인 필요"가 되지 않습니다.

<details><summary>힌트 보기</summary>

답에서 출처 번호는 `re.findall(r'[A-Z]+-[\w-]+#\d+', text)`로 찾을 수 있습니다. 1단계의 `answer()`는 결과의 `question`에 보낸 글(여기서는 프롬프트 전체)을 넣으므로, 질문으로 다시 바꿔 둡니다.

</details>

In [5]:
def answer_with_documents(question, k=4):
    """사내 문서를 검색해 근거를 붙여 답합니다. 1단계 answer()의 결과에 hits와 sources를 더합니다"""
    hits = searcher.search(question, k=k, where=is_valid)
    result = answer(rag_prompt(question, hits))
    result['question'] = question
    result['hits'] = [hit['chunk_id'] for hit in hits]
    result['sources'] = sorted(set(re.findall(r'[A-Z]+-[\w-]+#\d+', result['answer'])))
    if result['status'] == '완료' and '찾지 못했습니다' not in result['answer']:
        if not result['sources']:
            result['status'], result['reason'] = '확인 필요', '출처 없음'
        elif set(result['sources']) - set(result['hits']):
            result['status'], result['reason'] = '확인 필요', '검색하지 않은 출처'
    return result


for qid in ['E06', 'E24', 'E28', 'E41', 'E55']:
    result = answer_with_documents(questions[qid]['question'])
    print(f"[{result['status']}] {qid} {result['question']} (정답: {questions[qid]['answer']})")
    print(f"  → {result['answer'].strip().replace(chr(10), ' ')[:160]}")
    print(f"  검색 {result['hits']}, 출처 {result['sources']}, 사유 {result['reason']}")

[완료] E06 패키지 테스트에서 로트 수율이 몇 % 미만이면 로트를 홀드하나요? (정답: 97% 미만이면 홀드한다.)
  → 로트 수율이 **97% 미만**이면 로트를 홀드하고 테스트기술팀과 품질보증팀에 알립니다 [SOP-PT-001#4].
  검색 ['SOP-PT-001#4', 'QIR-2026-015#2', 'EDU-006#3', 'MTG-20260601#3'], 출처 ['SOP-PT-001#4'], 사유 end


[완료] E24 D-03 블레이드 때문에 생긴 칩핑 불량 때 블레이드의 누적 절단 거리는 얼마였나요? (정답: 11.5km.)
  → D-03 블레이드의 누적 절단 거리는 11.5km였습니다[QIR-2025-044#3]. 당시 교체 기준이었던 12km에는 이르지 않았지만, BG256은 웨이퍼가 150μm로 얇아 마모된 블레이드의 영향을 더 크게 받았습니다[QIR-2025-044#3].  참고로 검색 결과의 교체 기준이
  검색 ['QIR-2025-044#3', 'MAN-DS-001#3', 'QIR-2025-044#5', 'MTG-20260119#3'], 출처 ['MAN-DS-001#3', 'QIR-2025-044#3', 'QIR-2025-044#5'], 사유 end


[완료] E28 몰딩 컴파운드는 냉장 창고에서 꺼내 최소 몇 시간 해동하나요? (정답: 최소 6시간(SOP-MD-002 2차 개정).)
  → 냉장 창고에서 꺼낸 몰딩 컴파운드는 포장을 뜯지 않은 채 상온에서 최소 6시간 해동합니다 [SOP-MD-002#3]. 포장을 일찍 뜯으면 차가운 표면에 수분이 맺혀 기포와 미충전의 원인이 되므로 해동이 끝날 때까지 뜯지 않아야 합니다 [SOP-MD-002#3].
  검색 ['SOP-MD-002#4', 'SOP-MD-002#3', 'SOP-MD-002#2', 'SOP-MD-002#1'], 출처 ['SOP-MD-002#3'], 사유 end


[완료] E41 QX48의 몰딩 트랜스퍼 시간은 지금 몇 초인가요? (정답: 14초. 2026-11-16 회의에서 11월 23일부터 14초로 바꾸기로 했다. 작업 표준(SOP-MD-001 4차 개정)은 아직 12초로 남아 있다.)
  → QX48의 몰딩 트랜스퍼 시간은 검색 결과 중 가장 최신 문서 기준으로 **14초**입니다. 이 값은 11월 23일부터 적용하기로 결정되었습니다 [MTG-20261116#4].  검색 결과 사이에 내용 차이가 있습니다. 2025년 문서와 2026년 9월 문서는 트랜스퍼 시간을 12초로 
  검색 ['QIR-2025-031#5', 'QIR-2025-031#4', 'QIR-2026-019#4', 'MTG-20261116#4'], 출처 ['MTG-20261116#4', 'QIR-2025-031#4', 'QIR-2026-019#4'], 사유 end


[완료] E55 QX64의 몰딩 금형 온도는 몇 도인가요? (정답: 문서에 답이 없다. QX64의 공정 조건은 아직 작업 표준에 추가되지 않았다.)
  → 검색한 문서에서 찾지 못했습니다. QX64의 몰딩 금형 온도가 명시된 문서는 없습니다.  참고로, 몰딩 금형 클리닝 SOP에 "금형 온도는 175℃"라는 문장이 있지만 [SOP-MD-003#4], 이 문서는 QX64가 아닌 일반 몰딩 금형 클리닝 절차입니다. 따라서 이 온도를 QX64 
  검색 ['MTG-20260601#3', 'SOP-MD-003#4', 'EDU-013#1', 'EDU-013#2'], 출처 ['MTG-20260601#3', 'SOP-MD-003#4'], 사유 end
